<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/917_EPOv3_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit Tests

This is a **strong unit test suite** for several reasons:

### 1. Clear Scope

You correctly limit tests to **utilities**, not the graph.

```text
Data loading
Portfolio rollups
Risk rollups
Triggers
Health score
Report builder
```

Testing utilities instead of the orchestrator graph is exactly the right approach.

---

### 2. Deterministic Inputs

You use:

```python
tmp_path
mock snapshots
mock risk events
minimal datasets
```

This makes tests:

* reproducible
* fast
* independent of environment

Excellent practice.

---

### 3. Good Test Structure

Your tests follow a consistent pattern:

```python
Arrange
Act
Assert
```

Example:

```python
snap = [...]
r = build_portfolio_rollup(snap)
assert r["experiments_at_risk"] == 1
```

This is clear and readable.

---

# Coverage Assessment

Your coverage is already strong. Here's how it maps to the system.

| Component                | Covered |
| ------------------------ | ------- |
| JSON loading             | ✔       |
| Missing file behavior    | ✔       |
| Data validation warnings | ✔       |
| Experiment counting      | ✔       |
| Portfolio rollup logic   | ✔       |
| Trend classification     | ✔       |
| Risk rollup logic        | ✔       |
| Severity breakdown       | ✔       |
| Dominant risk type       | ✔       |
| Executive triggers       | ✔       |
| Health score             | ✔       |
| Report structure         | ✔       |
| Traceability fields      | ✔       |

This is **very comprehensive** for a utility layer.

---

# Strong Test Design Examples

### Orphan Risk Validation

This is a great test:

```python
assert any("E99" in w for w in out["validation_warnings"])
```

You’re verifying that the system **detects inconsistent datasets**.

This protects the system from **silent data corruption**.

---

### Trend Detection

This test is very good:

```python
assert r["portfolio_trend"] == "improving"
```

It confirms that the rollup logic correctly interprets:

```
risk ↓
velocity ↑
```

That’s exactly the kind of logic that should be tested.

---

### Health Score Floor

Another good edge-case test:

```python
assert score == 0
```

It ensures your score never becomes negative.

Edge cases like this are often forgotten.

---

# Two Small Improvements I Recommend

Your tests are already strong. These suggestions would just **increase robustness**.

---

# Improvement 1

## Test the Risk Narrative

Your report now includes a narrative like:

```
Data quality issues are currently the dominant portfolio risk...
```

You should test that this appears when expected.

Example:

```python
def test_build_experiment_report_risk_narrative():
    portfolio_rollup = {"portfolio_status": "healthy", "experiments_at_risk": 0}
    risk_rollup = {
        "open_count": 2,
        "by_severity_open": {"critical": 1},
        "dominant_risk_type": "data_quality_issue",
        "total_events": 3,
    }

    report = build_experiment_report(portfolio_rollup, risk_rollup, [])

    assert "dominant portfolio risk" in report
```

This protects the narrative logic.

---

# Improvement 2

## Test Portfolio Trend Sentence

Your new report includes:

```
Trend: Portfolio improving...
```

Add a simple test.

Example:

```python
def test_build_experiment_report_trend_sentence():
    portfolio_rollup = {
        "portfolio_status": "healthy",
        "portfolio_trend": "improving",
        "experiments_at_risk": 0,
    }

    risk_rollup = {"open_count": 0, "by_severity_open": {}, "dominant_risk_type": "none"}

    report = build_experiment_report(portfolio_rollup, risk_rollup, [])

    assert "Trend:" in report
```

This ensures the trend interpretation always appears.

---

# Minor Style Improvement

Instead of modifying `sys.path` in tests:

```python
root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
```

Most projects use:

```python
pytest -m
```

with the repo root as the working directory.

However, your approach is still valid if tests are run standalone.

---

# One Optional Advanced Test

You could add a **snapshot-style test** for the report.

Example:

```python
def test_report_snapshot():
    report = build_experiment_report(...)

    assert "Experimentation Portfolio — Executive Report" in report
    assert "## One view" in report
    assert "## Risk & governance" in report
```

This ensures the report format doesn’t accidentally change.

---

# Final Assessment

Your test suite demonstrates **excellent engineering maturity**.

It validates:

```text
data ingestion
portfolio intelligence
risk interpretation
governance triggers
executive report generation
```

Without needing to run the full agent graph.

That is exactly how a **well-tested orchestration system** should be structured.




In [ ]:
"""
Unit tests for EPOv3 orchestrator utilities.
Data loading, rollups, triggers, report builder. Use minimal or mock data; no graph.
"""
import json
import sys
from pathlib import Path

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pytest

from agents.epo_v3.orchestrator.utilities.data_loading import (
    load_json_file,
    load_all_experiment_data,
)
from agents.epo_v3.orchestrator.utilities.portfolio_rollup import (
    build_portfolio_rollup,
    build_risk_rollup,
    build_executive_triggers,
    compute_portfolio_health_score,
)
from agents.epo_v3.orchestrator.utilities.report import build_experiment_report


# ---- Data loading ----


def test_load_json_file_missing_returns_empty_list(tmp_path):
    out = load_json_file(tmp_path / "nonexistent.json")
    assert out == []


def test_load_json_file_valid_array(tmp_path):
    path = tmp_path / "data.json"
    path.write_text(json.dumps([{"id": "1"}, {"id": "2"}]))
    out = load_json_file(path)
    assert out == [{"id": "1"}, {"id": "2"}]


def test_load_json_file_empty_array(tmp_path):
    path = tmp_path / "data.json"
    path.write_text("[]")
    out = load_json_file(path)
    assert out == []


def test_load_all_experiment_data_returns_required_keys(tmp_path):
    (tmp_path / "experiment_runs.json").write_text("[]")
    (tmp_path / "experiment_portfolio_snapshots.json").write_text("[]")
    (tmp_path / "experiment_risk_events.json").write_text("[]")
    (tmp_path / "experiment_task_execution_logs.json").write_text("[]")
    out = load_all_experiment_data(str(tmp_path), tmp_path)
    assert "experiment_runs" in out
    assert "experiment_portfolio_snapshots" in out
    assert "experiment_risk_events" in out
    assert "experiment_task_execution_logs" in out
    assert "runs_count" in out
    assert "snapshots_count" in out
    assert "risk_events_count" in out
    assert "task_logs_count" in out
    assert "experiments_count" in out
    assert "data_snapshot_loaded_at" in out
    assert "validation_warnings" in out
    assert out["runs_count"] == 0
    assert out["experiments_count"] == 0


def test_load_all_experiment_data_counts_and_experiments(tmp_path):
    runs = [
        {"run_id": "XR1", "experiment_id": "E1"},
        {"run_id": "XR2", "experiment_id": "E1"},
        {"run_id": "XR3", "experiment_id": "E2"},
    ]
    (tmp_path / "experiment_runs.json").write_text(json.dumps(runs))
    (tmp_path / "experiment_portfolio_snapshots.json").write_text("[]")
    (tmp_path / "experiment_risk_events.json").write_text("[]")
    (tmp_path / "experiment_task_execution_logs.json").write_text("[]")
    out = load_all_experiment_data(str(tmp_path), tmp_path)
    assert out["runs_count"] == 3
    assert out["experiments_count"] == 2


def test_load_all_experiment_data_validation_warning_orphan_risk(tmp_path):
    (tmp_path / "experiment_runs.json").write_text(json.dumps([{"experiment_id": "E1"}]))
    (tmp_path / "experiment_portfolio_snapshots.json").write_text("[]")
    (tmp_path / "experiment_risk_events.json").write_text(
        json.dumps([{"event_id": "RE1", "experiment_id": "E99"}])
    )
    (tmp_path / "experiment_task_execution_logs.json").write_text("[]")
    out = load_all_experiment_data(str(tmp_path), tmp_path)
    assert any("E99" in w for w in out["validation_warnings"])


# ---- Portfolio rollup ----


def test_build_portfolio_rollup_empty():
    r = build_portfolio_rollup([])
    assert r["active_experiments"] == 0
    assert r["experiments_at_risk"] == 0
    assert r["portfolio_status"] == "unknown"


def test_build_portfolio_rollup_single_snapshot():
    snap = [
        {
            "snapshot_date": "2024-10-01",
            "active_experiments": 2,
            "experiments_at_risk": 1,
            "high_risk_experiments": 0,
            "portfolio_status": "healthy_with_watchlist",
            "learning_velocity_last_30d": 3,
        }
    ]
    r = build_portfolio_rollup(snap)
    assert r["active_experiments"] == 2
    assert r["experiments_at_risk"] == 1
    assert r["portfolio_status"] == "healthy_with_watchlist"
    assert r["portfolio_trend"] == "stable"
    assert r["delta_experiments_at_risk"] is None


def test_build_portfolio_rollup_two_snapshots_trend():
    snap = [
        {"snapshot_date": "2024-09-15", "experiments_at_risk": 2, "learning_velocity_last_30d": 1},
        {"snapshot_date": "2024-10-01", "experiments_at_risk": 1, "learning_velocity_last_30d": 3},
    ]
    r = build_portfolio_rollup(snap)
    assert r["snapshot_date"] == "2024-10-01"
    assert r["experiments_at_risk"] == 1
    assert r["delta_experiments_at_risk"] == -1
    assert r["delta_learning_velocity"] == 2
    assert r["portfolio_trend"] == "improving"


# ---- Risk rollup ----


def test_build_risk_rollup_empty():
    r = build_risk_rollup([])
    assert r["total_events"] == 0
    assert r["open_count"] == 0
    assert r["critical_count"] == 0
    assert r["by_severity"] == {}
    assert r["by_severity_open"] == {}
    assert r["dominant_risk_type"] == "none"


def test_build_risk_rollup_open_and_severity():
    events = [
        {"event_id": "RE1", "severity": "critical", "risk_type": "compliance_risk", "status": "blocked"},
        {"event_id": "RE2", "severity": "medium", "risk_type": "data_quality_issue", "status": "resolved"},
    ]
    r = build_risk_rollup(events)
    assert r["total_events"] == 2
    assert r["open_count"] == 1
    assert r["critical_count"] == 1
    assert r["by_severity"]["critical"] == 1
    assert r["by_severity"]["medium"] == 1
    assert r["by_severity_open"]["critical"] == 1
    assert "medium" not in r["by_severity_open"]
    assert r["dominant_risk_type"] == "compliance_risk"


# ---- Executive triggers ----


def test_build_executive_triggers_none():
    portfolio_rollup = {"experiments_at_risk": 0, "high_risk_experiments": 0, "learning_velocity_last_30d": 5}
    risk_rollup = {"open_count": 0, "critical_count": 0}
    t = build_executive_triggers(portfolio_rollup, risk_rollup)
    assert len(t) == 0


def test_build_executive_triggers_high_risk_and_critical():
    portfolio_rollup = {"experiments_at_risk": 0, "high_risk_experiments": 1, "learning_velocity_last_30d": 5}
    risk_rollup = {"open_count": 2, "critical_count": 1}
    t = build_executive_triggers(
        portfolio_rollup, risk_rollup, high_risk_experiments_critical=1, open_risk_events_critical=5
    )
    assert len(t) >= 2
    types = {x["trigger_type"] for x in t}
    assert "high_risk_experiments" in types
    assert "critical_risk_events" in types


# ---- Health score ----


def test_compute_portfolio_health_score():
    portfolio_rollup = {"experiments_at_risk": 1, "high_risk_experiments": 1}
    risk_rollup = {"open_count": 2}
    score = compute_portfolio_health_score(portfolio_rollup, risk_rollup)
    # 100 - 15*1 - 25*1 - 5*2 = 100 - 15 - 25 - 10 = 50
    assert score == 50


def test_compute_portfolio_health_score_floor_zero():
    portfolio_rollup = {"experiments_at_risk": 10, "high_risk_experiments": 5}
    risk_rollup = {"open_count": 10}
    score = compute_portfolio_health_score(portfolio_rollup, risk_rollup)
    assert score == 0


# ---- Report ----


def test_build_experiment_report_contains_sections():
    portfolio_rollup = {
        "portfolio_status": "healthy",
        "experiments_at_risk": 0,
        "high_risk_experiments": 0,
        "learning_velocity_last_30d": 3,
        "portfolio_health_score": 85,
    }
    risk_rollup = {"open_count": 0, "by_severity_open": {}, "dominant_risk_type": "none", "total_events": 0}
    report = build_experiment_report(portfolio_rollup, risk_rollup, [])
    assert "## One view" in report
    assert "## Portfolio health" in report
    assert "## Risk & governance" in report
    assert "## Traceability" in report
    assert "Portfolio status: healthy" in report
    assert "85 / 100" in report


def test_build_experiment_report_traceability_counts():
    portfolio_rollup = {"portfolio_status": "healthy", "experiments_at_risk": 0}
    risk_rollup = {"open_count": 0, "by_severity_open": {}, "dominant_risk_type": "none"}
    report = build_experiment_report(
        portfolio_rollup, risk_rollup, [],
        experiments_count=3, runs_count=10, risk_events_count=2, task_logs_count=10,
    )
    assert "3 experiments" in report
    assert "10 runs" in report
    assert "2 risk events" in report
